# Project 7 – Comparative Crash Pattern Analysis
## Signalized vs Unsignalized Intersections in Manhattan (2023-2024)

This notebook analyses motor vehicle crash patterns at signalized (traffic-signal-controlled)
and unsignalized intersections in Manhattan, New York City.

**Data sources:**
- NYC Motor Vehicle Collisions – Crashes (NYC Open Data)
- NYC Motor Vehicle Collisions – Vehicles (NYC Open Data)
- OpenStreetMap traffic signal node locations (via `osmnx`)

**Methodology:**
Crashes are spatially classified as *signalized* (within 30 m of an OSM `highway=traffic_signals` node)
or *unsignalized* (at an intersection but outside that buffer).

---

## 1. Setup & Imports

In [ ]:
import sys, os, logging, warnings
warnings.filterwarnings('ignore')

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_collection import load_crash_data, load_vehicle_data, fetch_traffic_signals
from src.preprocessing import (
    inspect_raw, filter_manhattan, clean_coordinates, deduplicate,
    add_severity, add_collision_type, add_temporal_features,
    create_intersection_key, classify_signalised, select_top_intersections,
    merge_vehicles,
)
from src.crash_analysis import run_analysis
from src.visualisation import generate_all_visualisations, setup_style
from src.collision_diagrams import generate_all_diagrams

print('All modules imported successfully.')

## 2. Data Loading

In [ ]:
crashes_raw = load_crash_data()
vehicles_raw = load_vehicle_data()

print(f'Crashes : {crashes_raw.shape}')
print(f'Vehicles: {vehicles_raw.shape}')
crashes_raw.head(3)

In [ ]:
vehicles_raw.head(3)

## 3. Preprocessing

Steps:
1. Filter to Manhattan, 2023-2024
2. Remove invalid coordinates and duplicates
3. Feature engineering (severity, collision type, temporal features)
4. Merge vehicle summary
5. Spatial classification (signalized / unsignalized)
6. Select top intersections

In [ ]:
# Step 1: Filter
crashes = filter_manhattan(crashes_raw)
print(f'After Manhattan + year filter: {len(crashes):,} records')

In [ ]:
# Step 2: Clean coordinates & deduplicate
crashes = clean_coordinates(crashes)
crashes = deduplicate(crashes)
print(f'After cleaning: {len(crashes):,} records')

In [ ]:
# Step 3: Feature engineering
crashes = add_severity(crashes)
crashes = add_collision_type(crashes)
crashes = add_temporal_features(crashes)
crashes = create_intersection_key(crashes)

print('Severity distribution:')
print(crashes['severity'].value_counts())
print()
print('Collision type distribution (top 5):')
print(crashes['collision_type'].value_counts().head())

In [ ]:
# Step 4: Merge vehicle summary
crashes = merge_vehicles(crashes, vehicles_raw)
print(f'After vehicle merge: {crashes.shape}')

In [ ]:
# Step 5: Spatial classification
signals_gdf = fetch_traffic_signals()
print(f'Traffic signal nodes: {len(signals_gdf):,}')

crashes = classify_signalised(crashes, signals_gdf=signals_gdf)
print()
print('Intersection type distribution:')
print(crashes['intersection_type'].value_counts())

In [ ]:
# Step 6: Top intersections
signalized, unsignalized = select_top_intersections(crashes)
print(f'Top signalized crashes  : {len(signalized):,}')
print(f'Top unsignalized crashes: {len(unsignalized):,}')

In [ ]:
# Save processed data
from src.data_collection import PROCESSED_DATA_DIR
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
crashes.to_csv(PROCESSED_DATA_DIR / 'manhattan_crashes_classified.csv', index=False)
signalized.to_csv(PROCESSED_DATA_DIR / 'top_signalized.csv', index=False)
unsignalized.to_csv(PROCESSED_DATA_DIR / 'top_unsignalized.csv', index=False)
print('Processed data saved.')

## 4. Exploratory Analysis

In [ ]:
# Quick overview of the classified dataset
intersection_crashes = crashes[crashes['intersection_type'].isin(['Signalized', 'Unsignalized'])]
print(f'Total intersection crashes: {len(intersection_crashes):,}')
print()
print(intersection_crashes.groupby('intersection_type')['severity'].value_counts())

In [ ]:
# Top intersections by crash count
for itype in ['Signalized', 'Unsignalized']:
    print(f'\n--- Top 5 {itype} Intersections ---')
    sub = intersection_crashes[intersection_crashes['intersection_type'] == itype]
    print(sub['intersection_key'].value_counts().head(5))

## 5. Crash Pattern Analysis

In [ ]:
results = run_analysis(crashes)
print('Analysis tables generated:')
for k, v in results.items():
    print(f'  {k}: {v.shape}')

In [ ]:
# Summary statistics
results['summary']

In [ ]:
# Chi-square tests
results['chi_square']

In [ ]:
# Collision type breakdown
results['collision_type']

In [ ]:
# Severity distribution
results['severity']

## 6. Visualisations

In [ ]:
paths = generate_all_visualisations(crashes, results)
print(f'Generated {len(paths)} visualisation files.')
for p in paths:
    print(f'  {p.name}')

In [ ]:
# Display key charts inline
from IPython.display import Image, display
from src.data_collection import PROJECT_ROOT

for img_name in ['crash_frequency_by_type', 'collision_type_comparison',
                 'severity_distribution', 'time_of_day_distribution',
                 'day_of_week_distribution', 'heatmap_signalized']:
    img_path = PROJECT_ROOT / 'outputs' / 'graphs' / f'{img_name}.png'
    if img_path.exists():
        print(f'\n--- {img_name} ---')
        display(Image(filename=str(img_path), width=700))

## 7. Collision Diagrams

In [ ]:
diagram_paths = generate_all_diagrams(signalized, unsignalized)
print(f'Generated {len(diagram_paths)} collision diagrams.')
for p in diagram_paths:
    print(f'  {p.name}')

In [ ]:
# Display first diagram from each type
sig_diagrams = [p for p in diagram_paths if 'signalized_' in p.name and 'unsignalized' not in p.name]
unsig_diagrams = [p for p in diagram_paths if 'unsignalized_' in p.name]

if sig_diagrams:
    print('--- Sample Signalized Collision Diagram ---')
    display(Image(filename=str(sig_diagrams[0]), width=600))

if unsig_diagrams:
    print('--- Sample Unsignalized Collision Diagram ---')
    display(Image(filename=str(unsig_diagrams[0]), width=600))

## 8. Comparative Analysis

### Key Findings

In [ ]:
summary = results['summary']
chi_sq = results['chi_square']

print('='*60)
print('COMPARATIVE SUMMARY')
print('='*60)
for _, row in summary.iterrows():
    print(f"\n{row['intersection_type']}:")
    print(f"  Total crashes        : {int(row['total_crashes']):,}")
    print(f"  Unique intersections : {int(row['unique_intersections']):,}")
    print(f"  Fatal                : {int(row['fatal'])} ({row['fatal_rate_pct']:.1f}%)")
    print(f"  Injury               : {int(row['injury'])} ({row['injury_rate_pct']:.1f}%)")
    print(f"  No Injury            : {int(row['no_injury'])}")

print('\n' + '='*60)
print('CHI-SQUARE TESTS (α = 0.05)')
print('='*60)
for _, row in chi_sq.iterrows():
    sig = '✓ Significant' if row['significant_005'] else '✗ Not significant'
    print(f"  {row['column']:30s}  χ²={row['chi2']:10.2f}  p={row['p_value']:.6f}  {sig}")

## 9. Summary

This analysis compared crash patterns at signalized and unsignalized intersections in Manhattan
using 2023-2024 NYC Open Data and OpenStreetMap traffic signal locations.

**Key results:**
- Crash frequency, severity, collision type, temporal, and contributing factor distributions
  were compared between the two intersection types.
- Chi-square tests assessed statistical significance of distributional differences.
- Collision diagrams provided schematic visual summaries of crash patterns at
  representative intersections.
- An interactive map enables spatial exploration of crash locations.

**Limitations:**
- Signal classification depends on OSM data completeness and a 30 m buffer heuristic.
- Collision diagrams are schematic; exact vehicle trajectories are not available.
- Crash reporting may be incomplete for minor incidents.